# Transfer Function Precompensation

This notebook keeps transfer-function work outside `OverlayController`. It loads measured S21 data, interpolates it to the waveform FFT bins, applies a regularized inverse, normalizes the result to `int16`, and optionally loads the precompensated waveform to DAC0.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from firmware import OverlayController
from firmware import signals

In [ ]:
ol = OverlayController()
info = ol.info()

DAC_SR = float(info["rfdc"]["dac0_sampling_rate_gsps"]) * 1e9
BUF_LEN = int(info["dac0"]["bram_int16_samples"])
DAC_AMP = int(np.iinfo(np.int16).max)

DAC_SR, BUF_LEN, DAC_AMP

In [ ]:
def to_int16(waveform, peak=DAC_AMP):
    data = np.asarray(waveform, dtype=float)
    return np.round(np.clip(data, -float(peak), float(peak))).astype(np.int16)


def normalize_to_dac(x, amp_limit):
    data = np.asarray(x, dtype=float)
    peak = np.max(np.abs(data)) if data.size else 0.0
    if peak == 0.0:
        return np.zeros(data.shape, dtype=np.int16)
    return to_int16(data * (float(amp_limit) / peak), peak=amp_limit)


def load_transfer_function_csv(csv_path):
    data = np.loadtxt(csv_path, delimiter=",", skiprows=1)
    f_hz = data[:, 0] * 1e9
    s21_db = data[:, 1]
    h_mag = 10 ** ((s21_db - 30) / 20.0)
    return f_hz, h_mag


def interp_transfer_to_rfft_bins(f_tf_hz, h_tf, n, fs):
    f_bins = np.fft.rfftfreq(n, d=1.0 / fs)
    h_bins = np.interp(f_bins, f_tf_hz, h_tf, left=h_tf[0], right=h_tf[-1])
    return f_bins, h_bins


def regularized_inverse(h, reg=0.02, floor=1e-3):
    h_abs = np.maximum(np.abs(h), floor)
    return np.conj(h) / (h_abs**2 + reg**2)

In [ ]:
def plot_signal(sig, sample_rate, title, samples=4096):
    view = np.asarray(sig)[:samples]
    t_us = np.arange(view.size) / sample_rate * 1e6
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(t_us, view)
    ax.set_title(title)
    ax.set_xlabel("Time (us)")
    ax.set_ylabel("DAC code")
    ax.grid(True, alpha=0.3)
    return ax


def plot_rfft(sig, sample_rate, title):
    data = np.asarray(sig, dtype=float)
    spectrum = np.fft.rfft(data)
    f_mhz = np.fft.rfftfreq(data.size, d=1.0 / sample_rate) / 1e6
    mag_db = 20 * np.log10(np.maximum(np.abs(spectrum), 1e-12))
    fig, ax = plt.subplots(figsize=(11, 3))
    ax.plot(f_mhz, mag_db)
    ax.set_title(title)
    ax.set_xlabel("Frequency (MHz)")
    ax.set_ylabel("Magnitude (dB)")
    ax.grid(True, alpha=0.3)
    return ax

## Generate Base Waveform

In [ ]:
freq_hz = 75e6
base_waveform = to_int16(
    signals.sawtooth(freq_hz=freq_hz, sample_rate=DAC_SR, num_samples=BUF_LEN, amplitude=0.8 * DAC_AMP)
)

plot_signal(base_waveform, DAC_SR, f"Base sawtooth: {freq_hz / 1e6:.1f} MHz")
plot_rfft(base_waveform, DAC_SR, "Base spectrum");

## Load and Interpolate Transfer Function

Place the measured CSV next to this notebook or adjust `csv_path`. The expected CSV format is frequency in GHz in column 0 and S21 in dB in column 1.

In [ ]:
csv_path = Path("S21_digitized_native.csv")
if not csv_path.exists():
    raise FileNotFoundError(f"Transfer-function CSV not found: {csv_path.resolve()}")

f_tf_hz, h_mag = load_transfer_function_csv(csv_path)
f_bins, h_bins = interp_transfer_to_rfft_bins(f_tf_hz, h_mag, n=len(base_waveform), fs=DAC_SR)

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(f_tf_hz / 1e9, h_mag, label="measured")
ax.plot(f_bins / 1e9, h_bins, label="rfft bins", alpha=0.7)
ax.set_title("Transfer function magnitude")
ax.set_xlabel("Frequency (GHz)")
ax.set_ylabel("Linear magnitude")
ax.grid(True, alpha=0.3)
ax.legend();

## Apply Regularized Inverse

In [ ]:
h_inv = regularized_inverse(h_bins, reg=0.02)

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(f_bins / 1e9, np.abs(h_inv))
ax.set_title("Regularized inverse magnitude")
ax.set_xlabel("Frequency (GHz)")
ax.set_ylabel("Linear gain")
ax.grid(True, alpha=0.3);

In [ ]:
base_fft = np.fft.rfft(base_waveform.astype(float))
pre_fft = base_fft * h_inv
precomp_float = np.fft.irfft(pre_fft, n=len(base_waveform))
precomp_waveform = normalize_to_dac(precomp_float, amp_limit=0.8 * DAC_AMP)

plot_signal(precomp_waveform, DAC_SR, "Precompensated waveform")
plot_rfft(precomp_waveform, DAC_SR, "Precompensated spectrum");

## Optional: Load Precompensated Waveform

In [ ]:
ol.dac0.load_waveform(precomp_waveform)
ol.info()

In [ ]:
ol.dac0.enable()
ol.dac0.is_enabled()

In [ ]:
ol.dac0.disable()
ol.info()